In [0]:
# ============================================================
# STEP 23.1 — Load Curated FMCG Sales Data
# ============================================================

sales_df = spark.table("fmcg.gold.fact_orders")

print("FMCG Sales data loaded successfully.")
print("Total rows:", sales_df.count())

display(sales_df.limit(10))

FMCG Sales data loaded successfully.
Total rows: 128210


date,product_code,customer_code,sold_quantity
2024-01-01,ARCHDDE20D,70002017,161
2024-01-01,ARCH158F41,70002017,133
2024-01-01,ARCHAFF0E4,70002017,76
2024-01-01,ARCH6B94F7,70002017,92
2024-01-01,ARCH5D1FE7,70002017,117
2024-01-01,ARCH7B49A9,70002017,36
2024-01-01,ARCH497D34,70002017,98
2024-01-01,ARCHE71D79,70002017,156
2024-01-01,BADM88C384,70002017,28
2024-01-01,BADMA5EBA3,70002017,33


In [0]:
# ============================================================
# STEP 23.2 — Inspect Fact Orders Schema
# ============================================================

sales_df.printSchema()

print("Columns available:")
for col in sales_df.columns:
    print("-", col)

root
 |-- date: date (nullable = true)
 |-- product_code: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- sold_quantity: long (nullable = true)

Columns available:
- date
- product_code
- customer_code
- sold_quantity


In [0]:
# ============================================================
# STEP 23.3 — Create Business-Friendly AI Dataset using PySpark
# ============================================================

from pyspark.sql.functions import (
    year,
    month,
    date_format,
    col
)

# Load Gold tables
fact_df = spark.table("fmcg.gold.fact_orders")
customer_df = spark.table("fmcg.gold.dim_customers")
product_df = spark.table("fmcg.gold.dim_products")
price_df = spark.table("fmcg.gold.dim_gross_price")

# Join Fact Orders with Customer, Product and Price dimensions
ai_sales_df = (
    fact_df.alias("f")
    
    .join(
        customer_df.alias("c"),
        col("f.customer_code") == col("c.customer_code"),
        "left"
    )
    
    .join(
        product_df.alias("p"),
        col("f.product_code") == col("p.product_code"),
        "left"
    )
    
    .join(
        price_df.alias("gp"),
        (
            (col("f.product_code") == col("gp.product_code")) &
            (year(col("f.date")) == col("gp.year"))
        ),
        "left"
    )
    
    .select(
        col("f.date").alias("date"),
        year(col("f.date")).alias("year"),
        month(col("f.date")).alias("month_number"),
        date_format(col("f.date"), "MMMM").alias("month_name"),
        date_format(col("f.date"), "yyyy-MM").alias("month_year"),

        col("f.customer_code").alias("customer_code"),
        col("c.customer").alias("customer"),
        col("c.market").alias("market"),
        col("c.platform").alias("platform"),
        col("c.channel").alias("channel"),

        col("f.product_code").alias("product_code"),
        col("p.product").alias("product_name"),
        col("p.category").alias("category"),

        col("f.sold_quantity").alias("sold_quantity"),

        col("gp.price_inr").alias("price_inr"),

        (
            col("f.sold_quantity") * col("gp.price_inr")
        ).alias("revenue_inr")
    )
)

print("AI business dataset created successfully.")
print("Total rows:", ai_sales_df.count())

display(ai_sales_df.limit(10))

AI business dataset created successfully.
Total rows: 128210


date,year,month_number,month_name,month_year,customer_code,customer,market,platform,channel,product_code,product_name,category,sold_quantity,price_inr,revenue_inr
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCHDDE20D,PX Carbon Arrow Set,Arrows,161,2750,442750
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH158F41,LX Leather Finger Tab,Finger Tab,133,5740,763420
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCHAFF0E4,AX Precision Recurve Bow,Bow,76,4610,350360
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH6B94F7,AX Precision Recurve Bow,Bow,92,3910,359720
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH5D1FE7,BX Bow Stringing Tool,Bow Stringer,117,1610,188370
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH7B49A9,NX Archery Arm Guard,Arm Guard,36,1610,57960
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH497D34,TX Foam Archery Target,Target,98,1100,107800
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCHE71D79,NX Archery Arm Guard,Arm Guard,156,5830,909480
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,BADM88C384,AX PowerDrive Badminton Racket,Badminton Racket,28,790,22120
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,BADMA5EBA3,BX AeroLite Badminton Racket,Badminton Racket,33,5910,195030


In [0]:
# ============================================================
# STEP 23.4 — Validate AI Business Dataset
# ============================================================

from pyspark.sql.functions import (
    col,
    sum,
    when,
    count
)

# ------------------------------------------------------------
# 1. Check NULL values in important business columns
# ------------------------------------------------------------

null_check_df = ai_sales_df.select(
    sum(when(col("date").isNull(), 1).otherwise(0)).alias("null_dates"),
    sum(when(col("customer").isNull(), 1).otherwise(0)).alias("null_customers"),
    sum(when(col("product_name").isNull(), 1).otherwise(0)).alias("null_products"),
    sum(when(col("category").isNull(), 1).otherwise(0)).alias("null_categories"),
    sum(when(col("sold_quantity").isNull(), 1).otherwise(0)).alias("null_quantities"),
    sum(when(col("price_inr").isNull(), 1).otherwise(0)).alias("null_prices"),
    sum(when(col("revenue_inr").isNull(), 1).otherwise(0)).alias("null_revenue")
)

display(null_check_df)


# ------------------------------------------------------------
# 2. Check negative / zero quantities
# ------------------------------------------------------------

quantity_check_df = ai_sales_df.select(
    sum(
        when(col("sold_quantity") <= 0, 1).otherwise(0)
    ).alias("non_positive_quantity")
)

display(quantity_check_df)


# ------------------------------------------------------------
# 3. Check revenue calculation
# ------------------------------------------------------------

revenue_check_df = ai_sales_df.select(
    count("*").alias("total_rows"),
    sum("sold_quantity").alias("total_quantity"),
    sum("revenue_inr").alias("total_revenue")
)

display(revenue_check_df)


# ------------------------------------------------------------
# 4. Display sample business records
# ------------------------------------------------------------

display(
    ai_sales_df.select(
        "date",
        "customer",
        "product_name",
        "category",
        "channel",
        "sold_quantity",
        "price_inr",
        "revenue_inr"
    ).limit(10)
)

null_dates,null_customers,null_products,null_categories,null_quantities,null_prices,null_revenue
14,3182,0,0,0,14,14


non_positive_quantity
0


total_rows,total_quantity,total_revenue
128210,34598880,105373190449


date,customer,product_name,category,channel,sold_quantity,price_inr,revenue_inr
2024-01-01,Sporty Essentials,PX Carbon Arrow Set,Arrows,Retailer,161,2750,442750
2024-01-01,Sporty Essentials,LX Leather Finger Tab,Finger Tab,Retailer,133,5740,763420
2024-01-01,Sporty Essentials,AX Precision Recurve Bow,Bow,Retailer,76,4610,350360
2024-01-01,Sporty Essentials,AX Precision Recurve Bow,Bow,Retailer,92,3910,359720
2024-01-01,Sporty Essentials,BX Bow Stringing Tool,Bow Stringer,Retailer,117,1610,188370
2024-01-01,Sporty Essentials,NX Archery Arm Guard,Arm Guard,Retailer,36,1610,57960
2024-01-01,Sporty Essentials,TX Foam Archery Target,Target,Retailer,98,1100,107800
2024-01-01,Sporty Essentials,NX Archery Arm Guard,Arm Guard,Retailer,156,5830,909480
2024-01-01,Sporty Essentials,AX PowerDrive Badminton Racket,Badminton Racket,Retailer,28,790,22120
2024-01-01,Sporty Essentials,BX AeroLite Badminton Racket,Badminton Racket,Retailer,33,5910,195030


In [0]:
# ============================================================
# STEP 23.5 — Create AI-Friendly Summary DataFrames
# ============================================================

from pyspark.sql.functions import (
    sum,
    countDistinct,
    desc,
    asc,
    round
)

# ------------------------------------------------------------
# 1. Monthly Sales Summary
# ------------------------------------------------------------

ai_monthly_df = (
    ai_sales_df
    .groupBy(
        "year",
        "month_number",
        "month_name",
        "month_year"
    )
    .agg(
        sum("sold_quantity").alias("total_quantity"),
        sum("revenue_inr").alias("total_revenue"),
        countDistinct("customer_code").alias("unique_customers"),
        countDistinct("product_code").alias("unique_products")
    )
    .orderBy("year", "month_number")
)


# ------------------------------------------------------------
# 2. Category Sales Summary
# ------------------------------------------------------------

ai_category_df = (
    ai_sales_df
    .groupBy("category")
    .agg(
        sum("sold_quantity").alias("total_quantity"),
        sum("revenue_inr").alias("total_revenue"),
        countDistinct("product_code").alias("unique_products"),
        countDistinct("customer_code").alias("unique_customers")
    )
    .orderBy(desc("total_revenue"))
)


# ------------------------------------------------------------
# 3. Product Sales Summary
# ------------------------------------------------------------

ai_product_df = (
    ai_sales_df
    .groupBy(
        "product_code",
        "product_name",
        "category"
    )
    .agg(
        sum("sold_quantity").alias("total_quantity"),
        sum("revenue_inr").alias("total_revenue"),
        countDistinct("customer_code").alias("unique_customers")
    )
    .orderBy(desc("total_revenue"))
)


# ------------------------------------------------------------
# 4. Customer Sales Summary
# ------------------------------------------------------------

ai_customer_df = (
    ai_sales_df
    .groupBy(
        "customer_code",
        "customer",
        "market",
        "platform",
        "channel"
    )
    .agg(
        sum("sold_quantity").alias("total_quantity"),
        sum("revenue_inr").alias("total_revenue"),
        countDistinct("product_code").alias("unique_products")
    )
    .orderBy(desc("total_revenue"))
)


# ------------------------------------------------------------
# 5. Channel Sales Summary
# ------------------------------------------------------------

ai_channel_df = (
    ai_sales_df
    .groupBy("channel")
    .agg(
        sum("sold_quantity").alias("total_quantity"),
        sum("revenue_inr").alias("total_revenue"),
        countDistinct("customer_code").alias("unique_customers")
    )
    .orderBy(desc("total_revenue"))
)


print("AI summary datasets created successfully.")

print("Monthly rows:", ai_monthly_df.count())
print("Category rows:", ai_category_df.count())
print("Product rows:", ai_product_df.count())
print("Customer rows:", ai_customer_df.count())
print("Channel rows:", ai_channel_df.count())

AI summary datasets created successfully.
Monthly rows: 24
Category rows: 117
Product rows: 406
Customer rows: 57
Channel rows: 4


In [0]:
# ============================================================
# STEP 23.6 — Test Business Questions
# ============================================================

from pyspark.sql.functions import sum, desc, round

# ------------------------------------------------------------
# Question 1:
# What is the total revenue?
# ------------------------------------------------------------

total_revenue = (
    ai_sales_df
    .agg(
        sum("revenue_inr").alias("total_revenue")
    )
)

print("Q1: What is the total revenue?")
display(total_revenue)


# ------------------------------------------------------------
# Question 2:
# Which category generated the most revenue?
# ------------------------------------------------------------

print("Q2: Which category generated the most revenue?")

display(
    ai_category_df
    .select(
        "category",
        "total_revenue"
    )
    .orderBy(desc("total_revenue"))
    .limit(10)
)


# ------------------------------------------------------------
# Question 3:
# Which products generated the most revenue?
# ------------------------------------------------------------

print("Q3: Which products generated the most revenue?")

display(
    ai_product_df
    .select(
        "product_code",
        "product_name",
        "category",
        "total_revenue"
    )
    .orderBy(desc("total_revenue"))
    .limit(10)
)


# ------------------------------------------------------------
# Question 4:
# Which customers generated the most revenue?
# ------------------------------------------------------------

print("Q4: Which customers generated the most revenue?")

display(
    ai_customer_df
    .select(
        "customer_code",
        "customer",
        "market",
        "channel",
        "total_revenue"
    )
    .orderBy(desc("total_revenue"))
    .limit(10)
)


# ------------------------------------------------------------
# Question 5:
# Which channel generated the most revenue?
# ------------------------------------------------------------

print("Q5: Which channel generated the most revenue?")

display(
    ai_channel_df
    .select(
        "channel",
        "total_revenue"
    )
    .orderBy(desc("total_revenue"))
)

Q1: What is the total revenue?


total_revenue
105373190449


Q2: Which category generated the most revenue?


category,total_revenue
Football,10345489260
Cricket Batting Gloves,7982130900
Dumbbells,7621833220
Cricket Pads,6212228820
Football Shoes,5525410970
Cricket Ball,5320029340
Cricket Kit Bag,5124440500
Cricket Helmet,4949226220
Cricket Bat,4546333630
Weight Bars,4145549530


Q3: Which products generated the most revenue?


product_code,product_name,category,total_revenue
WLFTB7DF2B,WL Steel Barbell,Weight Bars,3697666040
WLFT7AD84A,WL Hex Dumbbell,Dumbbells,3297536070
WLFT053057,WL Hex Dumbbell,Dumbbells,2192933600
WLFT52B261,WL Hex Dumbbell,Dumbbells,1914774390
CRIC35B3C7,PX Grip Cricket Batting Gloves,Cricket Batting Gloves,1502841190
CRIC538334,KX Blaze Kashmir Willow Cricket Bat,Cricket Bat,1412230160
CRIC8DF8D8,AX Guard Cricket Helmet,Cricket Helmet,1398319300
CRICBDB8DA,TRX Max Cricket Kit Bag,Cricket Kit Bag,1363891000
FOOT7DA9E9,RX Sprint Football Boots,Football Shoes,1331527340
FOOTF63A17,RX Sprint Football Boots,Football Shoes,1312814160


Q4: Which customers generated the most revenue?


customer_code,customer,market,channel,total_revenue
90002008,FitnessWorld,India,Retailer,9532997850
90002009,FastTrack Sports,India,Retailer,8205305690
90002016,Fitness Mania,India,Retailer,5960288480
90002013,Active Gear,India,Retailer,5874425360
90002005,Atlikon Official,India,Direct,5795650270
90002001,SportsHub,India,Retailer,5728477020
70002017,Sporty Essentials,India,Retailer,5588030530
90002004,Atlikon Sports India,India,Direct,5548912140
90002012,Ultimate Sports,India,Retailer,5443598040
70002018,PlayMore Sports,India,Retailer,5407843120


Q5: Which channel generated the most revenue?


channel,total_revenue
Retailer,82677359800
Direct,21307439590
Acquisition,1248247849
null,140143210


In [0]:
# ============================================================
# STEP 23.7 — Save AI-Ready Summary Tables
# ============================================================

# Monthly sales
ai_monthly_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fmcg.gold.ai_monthly_sales")


# Category sales
ai_category_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fmcg.gold.ai_category_sales")


# Product sales
ai_product_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fmcg.gold.ai_product_sales")


# Customer sales
ai_customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fmcg.gold.ai_customer_sales")


# Channel sales
ai_channel_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fmcg.gold.ai_channel_sales")


print("============================================================")
print("AI-READY GOLD TABLES CREATED SUCCESSFULLY")
print("============================================================")

print("1. fmcg.gold.ai_monthly_sales")
print("2. fmcg.gold.ai_category_sales")
print("3. fmcg.gold.ai_product_sales")
print("4. fmcg.gold.ai_customer_sales")
print("5. fmcg.gold.ai_channel_sales")

AI-READY GOLD TABLES CREATED SUCCESSFULLY
1. fmcg.gold.ai_monthly_sales
2. fmcg.gold.ai_category_sales
3. fmcg.gold.ai_product_sales
4. fmcg.gold.ai_customer_sales
5. fmcg.gold.ai_channel_sales


In [0]:
# ============================================================
# STEP 23.8 — Create AI Business Definitions
# ============================================================

from pyspark.sql.types import StructType, StructField, StringType

# Business definitions for the AI Assistant
business_definitions = [
    ("revenue_inr",
     "Total sales revenue in Indian Rupees (INR). Calculated as sold quantity multiplied by applicable gross price."),

    ("sold_quantity",
     "Total quantity of products sold."),

    ("price_inr",
     "Gross selling price of the product in Indian Rupees (INR)."),

    ("date",
     "Date on which the sale/order occurred."),

    ("year",
     "Calendar year of the sale."),

    ("month_number",
     "Numeric month of the sale, from 1 to 12."),

    ("month_name",
     "Name of the calendar month."),

    ("month_year",
     "Year-month representation of the sale, such as 2025-01."),

    ("customer_code",
     "Unique code identifying a customer."),

    ("customer",
     "Customer name."),

    ("market",
     "Market associated with the customer."),

    ("platform",
     "Sales platform associated with the customer/order."),

    ("channel",
     "Sales channel through which the product was sold."),

    ("product_code",
     "Unique code identifying a product."),

    ("product_name",
     "Name of the product."),

    ("category",
     "Product category."),

    ("total_revenue",
     "Aggregated sales revenue in Indian Rupees (INR)."),

    ("total_quantity",
     "Aggregated quantity of products sold."),

    ("unique_customers",
     "Number of distinct customers associated with the sales."),

    ("unique_products",
     "Number of distinct products associated with the sales.")
]

schema = StructType([
    StructField("column_name", StringType(), False),
    StructField("business_definition", StringType(), False)
])

ai_definitions_df = spark.createDataFrame(
    business_definitions,
    schema
)

print("AI business definitions created successfully.")

display(ai_definitions_df)

AI business definitions created successfully.


column_name,business_definition
revenue_inr,Total sales revenue in Indian Rupees (INR). Calculated as sold quantity multiplied by applicable gross price.
sold_quantity,Total quantity of products sold.
price_inr,Gross selling price of the product in Indian Rupees (INR).
date,Date on which the sale/order occurred.
year,Calendar year of the sale.
month_number,"Numeric month of the sale, from 1 to 12."
month_name,Name of the calendar month.
month_year,"Year-month representation of the sale, such as 2025-01."
customer_code,Unique code identifying a customer.
customer,Customer name.


In [0]:
# ============================================================
# STEP 23.9 — Save AI Business Definitions
# ============================================================

ai_definitions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fmcg.gold.ai_business_definitions")

print("AI business definitions saved successfully.")

AI business definitions saved successfully.


In [0]:
# ============================================================
# STEP 23.10 — Create FMCG AI Query Function
# ============================================================

from pyspark.sql.functions import (
    sum,
    desc,
    col
)

def fmcg_ai_query(question):

    question = question.lower().strip()

    # --------------------------------------------------------
    # 1. Total Revenue
    # --------------------------------------------------------

    if "total revenue" in question:

        result = (
            spark.table("fmcg.gold.ai_category_sales")
            .agg(
                sum("total_revenue").alias("total_revenue")
            )
        )

        print("Answer: Total Revenue")
        display(result)


    # --------------------------------------------------------
    # 2. Revenue by Category
    # --------------------------------------------------------

    elif "category" in question and "revenue" in question:

        result = (
            spark.table("fmcg.gold.ai_category_sales")
            .select(
                "category",
                "total_revenue"
            )
            .orderBy(desc("total_revenue"))
        )

        print("Answer: Revenue by Category")
        display(result)


    # --------------------------------------------------------
    # 3. Top Products
    # --------------------------------------------------------

    elif "product" in question and (
        "top" in question or
        "highest" in question or
        "most" in question
    ):

        result = (
            spark.table("fmcg.gold.ai_product_sales")
            .select(
                "product_code",
                "product_name",
                "category",
                "total_revenue"
            )
            .orderBy(desc("total_revenue"))
            .limit(10)
        )

        print("Answer: Top 10 Products by Revenue")
        display(result)


    # --------------------------------------------------------
    # 4. Top Customers
    # --------------------------------------------------------

    elif "customer" in question and (
        "top" in question or
        "highest" in question or
        "most" in question
    ):

        result = (
            spark.table("fmcg.gold.ai_customer_sales")
            .select(
                "customer_code",
                "customer",
                "market",
                "channel",
                "total_revenue"
            )
            .orderBy(desc("total_revenue"))
            .limit(10)
        )

        print("Answer: Top 10 Customers by Revenue")
        display(result)


    # --------------------------------------------------------
    # 5. Revenue by Channel
    # --------------------------------------------------------

    elif "channel" in question and "revenue" in question:

        result = (
            spark.table("fmcg.gold.ai_channel_sales")
            .select(
                "channel",
                "total_revenue"
            )
            .orderBy(desc("total_revenue"))
        )

        print("Answer: Revenue by Channel")
        display(result)


    # --------------------------------------------------------
    # 6. Monthly Revenue
    # --------------------------------------------------------

    elif "monthly" in question and "revenue" in question:

        result = (
            spark.table("fmcg.gold.ai_monthly_sales")
            .select(
                "month_year",
                "total_revenue"
            )
            .orderBy("month_year")
        )

        print("Answer: Monthly Revenue")
        display(result)


    # --------------------------------------------------------
    # 7. Unknown Question
    # --------------------------------------------------------

    else:

        print(
            "I can currently answer questions about "
            "total revenue, categories, products, customers, "
            "channels and monthly revenue."
        )

In [0]:
fmcg_ai_query("What is the total revenue?")

Answer: Total Revenue


total_revenue
105373190449


In [0]:
fmcg_ai_query("Which category has the highest revenue?")

Answer: Revenue by Category


category,total_revenue
Football,10345489260
Cricket Batting Gloves,7982130900
Dumbbells,7621833220
Cricket Pads,6212228820
Football Shoes,5525410970
Cricket Ball,5320029340
Cricket Kit Bag,5124440500
Cricket Helmet,4949226220
Cricket Bat,4546333630
Weight Bars,4145549530


In [0]:
# ============================================================
# STEP 23.11 — Enhanced FMCG AI Query Function
# ============================================================

from pyspark.sql.functions import (
    sum,
    desc,
    col
)


def fmcg_ai_query(question):

    question = question.lower().strip()

    # ========================================================
    # 1. Revenue for a Specific Year
    # ========================================================

    if "revenue" in question and "2025" in question:

        result = (
            spark.table("fmcg.gold.ai_monthly_sales")
            .filter(col("year") == 2025)
            .agg(
                sum("total_revenue").alias("revenue_2025")
            )
        )

        print("Answer: Revenue for 2025")
        display(result)


    # ========================================================
    # 2. Quantity for a Specific Year
    # ========================================================

    elif "quantity" in question and "2025" in question:

        result = (
            spark.table("fmcg.gold.ai_monthly_sales")
            .filter(col("year") == 2025)
            .agg(
                sum("total_quantity").alias("quantity_2025")
            )
        )

        print("Answer: Quantity Sold in 2025")
        display(result)


    # ========================================================
    # 3. Revenue for Beverages
    # ========================================================

    elif "beverages" in question and "revenue" in question:

        result = (
            spark.table("fmcg.gold.ai_category_sales")
            .filter(
                col("category").contains("Beverages")
            )
            .select(
                "category",
                "total_revenue"
            )
        )

        print("Answer: Beverages Revenue")
        display(result)


    # ========================================================
    # 4. Quantity for Beverages
    # ========================================================

    elif "beverages" in question and "quantity" in question:

        result = (
            spark.table("fmcg.gold.ai_category_sales")
            .filter(
                col("category").contains("Beverages")
            )
            .select(
                "category",
                "total_quantity"
            )
        )

        print("Answer: Beverages Quantity Sold")
        display(result)


    # ========================================================
    # 5. Total Revenue
    # ========================================================

    elif "total revenue" in question:

        result = (
            spark.table("fmcg.gold.ai_category_sales")
            .agg(
                sum("total_revenue").alias("total_revenue")
            )
        )

        print("Answer: Total Revenue")
        display(result)


    # ========================================================
    # 6. Revenue by Category
    # ========================================================

    elif "category" in question and "revenue" in question:

        result = (
            spark.table("fmcg.gold.ai_category_sales")
            .select(
                "category",
                "total_revenue"
            )
            .orderBy(desc("total_revenue"))
        )

        print("Answer: Revenue by Category")
        display(result)


    # ========================================================
    # 7. Top Products
    # ========================================================

    elif "product" in question and (
        "top" in question or
        "highest" in question or
        "most" in question
    ):

        result = (
            spark.table("fmcg.gold.ai_product_sales")
            .select(
                "product_code",
                "product_name",
                "category",
                "total_revenue"
            )
            .orderBy(desc("total_revenue"))
            .limit(10)
        )

        print("Answer: Top 10 Products by Revenue")
        display(result)


    # ========================================================
    # 8. Top Customers
    # ========================================================

    elif "customer" in question and (
        "top" in question or
        "highest" in question or
        "most" in question
    ):

        result = (
            spark.table("fmcg.gold.ai_customer_sales")
            .select(
                "customer_code",
                "customer",
                "market",
                "channel",
                "total_revenue"
            )
            .orderBy(desc("total_revenue"))
            .limit(10)
        )

        print("Answer: Top 10 Customers by Revenue")
        display(result)


    # ========================================================
    # 9. Revenue by Channel
    # ========================================================

    elif "channel" in question and "revenue" in question:

        result = (
            spark.table("fmcg.gold.ai_channel_sales")
            .select(
                "channel",
                "total_revenue"
            )
            .orderBy(desc("total_revenue"))
        )

        print("Answer: Revenue by Channel")
        display(result)


    # ========================================================
    # 10. Monthly Revenue
    # ========================================================

    elif "monthly" in question and "revenue" in question:

        result = (
            spark.table("fmcg.gold.ai_monthly_sales")
            .select(
                "month_year",
                "total_revenue"
            )
            .orderBy("month_year")
        )

        print("Answer: Monthly Revenue")
        display(result)


    # ========================================================
    # 11. Unsupported Question
    # ========================================================

    else:

        print(
            "I can currently answer questions about "
            "revenue, quantity, years, categories, products, "
            "customers, channels and monthly sales."
        )

In [0]:
fmcg_ai_query("What was the revenue in 2025?")
fmcg_ai_query("What was the quantity sold in 2025?")
fmcg_ai_query("How much revenue did Beverages generate?")

Answer: Revenue for 2025


revenue_2025
70302199119


Answer: Quantity Sold in 2025


quantity_2025
26001146


Answer: Beverages Revenue


category,total_revenue


In [0]:
# ============================================================
# STEP 23.12 — Create AI Assistant Master Dataset
# ============================================================

# Select only the business fields needed by the assistant

ai_master_df = (
    ai_sales_df
    .select(
        "date",
        "year",
        "month_number",
        "month_name",
        "month_year",
        "customer_code",
        "customer",
        "market",
        "platform",
        "channel",
        "product_code",
        "product_name",
        "category",
        "sold_quantity",
        "price_inr",
        "revenue_inr"
    )
)

print("AI master dataset created successfully.")
print("Total rows:", ai_master_df.count())

display(ai_master_df.limit(10))

AI master dataset created successfully.
Total rows: 128210


date,year,month_number,month_name,month_year,customer_code,customer,market,platform,channel,product_code,product_name,category,sold_quantity,price_inr,revenue_inr
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCHDDE20D,PX Carbon Arrow Set,Arrows,161,2750,442750
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH158F41,LX Leather Finger Tab,Finger Tab,133,5740,763420
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCHAFF0E4,AX Precision Recurve Bow,Bow,76,4610,350360
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH6B94F7,AX Precision Recurve Bow,Bow,92,3910,359720
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH5D1FE7,BX Bow Stringing Tool,Bow Stringer,117,1610,188370
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH7B49A9,NX Archery Arm Guard,Arm Guard,36,1610,57960
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCH497D34,TX Foam Archery Target,Target,98,1100,107800
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,ARCHE71D79,NX Archery Arm Guard,Arm Guard,156,5830,909480
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,BADM88C384,AX PowerDrive Badminton Racket,Badminton Racket,28,790,22120
2024-01-01,2024,1,January,2024-01,70002017,Sporty Essentials,India,Brick & Mortar,Retailer,BADMA5EBA3,BX AeroLite Badminton Racket,Badminton Racket,33,5910,195030


In [0]:
# ============================================================
# STEP 23.12B — Save AI Assistant Master Dataset
# ============================================================

ai_master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fmcg.gold.ai_master_sales")

print("AI master Gold table created successfully.")
print("Table: fmcg.gold.ai_master_sales")

AI master Gold table created successfully.
Table: fmcg.gold.ai_master_sales
